# Image Segmentation with YOLOv8 for Pothole Detection

This notebook demonstrates **instance segmentation** using a **YOLOv8-seg** model,
trained on the [Pothole Image Segmentation Dataset](https://www.kaggle.com/datasets/raunakkesharwani/pothole-image-segmentation-dataset).

The dataset provides YOLO-format polygon segmentation labels (normalized vertex coordinates),
which map naturally to the Ultralytics training pipeline.

**Pipeline overview:**
1. Explore the dataset structure and label format
2. Configure a `data.yaml` pointing to the correct paths
3. Fine-tune a YOLOv8n-seg pretrained model on pothole images
4. Evaluate with mAP and IoU metrics
5. Visualize segmentation mask predictions

**Key libraries:** Ultralytics (YOLOv8), OpenCV, Matplotlib

In [ ]:
# ============================================================
# Cell 1 - Install & Import
# ============================================================

!pip install -q ultralytics

import os
import glob
import random
import yaml
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

from ultralytics import YOLO

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Data Download (runtime fallback when dataset not mounted)
# ============================================================
import os, glob

KAGGLE_DATA_ROOT = "/kaggle/input/pothole-image-segmentation-dataset/dataset"

if not os.path.exists(KAGGLE_DATA_ROOT):
    print("Dataset not mounted via /kaggle/input, downloading...")
    os.makedirs("/kaggle/working/data", exist_ok=True)
    os.system("kaggle datasets download -d raunakkesharwani/pothole-image-segmentation-dataset -p /kaggle/working/data --unzip")
    # Find the dataset folder with train/val
    candidates = glob.glob("/kaggle/working/data/**/train", recursive=True)
    if candidates:
        KAGGLE_DATA_ROOT = str(os.path.dirname(candidates[0]))
    else:
        KAGGLE_DATA_ROOT = "/kaggle/working/data"
    print(f"Data downloaded. Root: {KAGGLE_DATA_ROOT}")
else:
    print(f"Dataset found at: {KAGGLE_DATA_ROOT}")


In [ ]:
# ============================================================
# Cell 2 - Configuration & Dataset Paths
# ============================================================

# Dataset root on Kaggle
# DATA_ROOT set by download cell above
DATA_ROOT = KAGGLE_DATA_ROOT

TRAIN_IMAGES = os.path.join(DATA_ROOT, "train", "images")
TRAIN_LABELS = os.path.join(DATA_ROOT, "train", "labels")
VAL_IMAGES   = os.path.join(DATA_ROOT, "val", "images")
VAL_LABELS   = os.path.join(DATA_ROOT, "val", "labels")

# Verify paths exist
for p in [TRAIN_IMAGES, TRAIN_LABELS, VAL_IMAGES, VAL_LABELS]:
    exists = os.path.isdir(p)
    print(f"{'OK' if exists else 'MISSING':>7}  {p}")

# Count files
n_train_img = len(glob.glob(os.path.join(TRAIN_IMAGES, "*.jpg")))
n_train_lbl = len(glob.glob(os.path.join(TRAIN_LABELS, "*.txt")))
n_val_img   = len(glob.glob(os.path.join(VAL_IMAGES, "*.jpg")))
n_val_lbl   = len(glob.glob(os.path.join(VAL_LABELS, "*.txt")))

print(f"\nTrain: {n_train_img} images, {n_train_lbl} labels")
print(f"Val:   {n_val_img} images, {n_val_lbl} labels")

In [ ]:
# ============================================================
# Cell 3 - Explore the Label Format
# ============================================================

# YOLO segmentation labels: each line is
#   class_id x1 y1 x2 y2 x3 y3 ... (normalized 0-1 coordinates)
# This defines a polygon mask for each object instance.

sample_labels = sorted(glob.glob(os.path.join(TRAIN_LABELS, "*.txt")))[:3]

for lbl_path in sample_labels:
    print(f"\n--- {os.path.basename(lbl_path)} ---")
    with open(lbl_path, "r") as f:
        lines = f.readlines()
    for line in lines[:3]:  # show first 3 objects per file
        tokens = line.strip().split()
        class_id = tokens[0]
        n_points = (len(tokens) - 1) // 2
        print(f"  class={class_id}, polygon vertices={n_points}, "
              f"first coords: {' '.join(tokens[1:7])}...")
    if len(lines) > 3:
        print(f"  ... ({len(lines)} objects total)")

In [ ]:
# ============================================================
# Cell 4 - Visualize Sample Images with Polygon Labels
# ============================================================

def draw_yolo_polygons(image_path, label_path):
    """Draw YOLO polygon segmentation labels on an image."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    overlay = img.copy()

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                tokens = line.strip().split()
                if len(tokens) < 5:
                    continue
                coords = list(map(float, tokens[1:]))
                # Convert normalized coords to pixel coords
                points = []
                for i in range(0, len(coords), 2):
                    px = int(coords[i] * w)
                    py = int(coords[i + 1] * h)
                    points.append([px, py])
                pts = np.array(points, dtype=np.int32)
                # Fill polygon with semi-transparent color
                cv2.fillPoly(overlay, [pts], (255, 0, 0))
                cv2.polylines(img, [pts], True, (255, 0, 0), 2)

    # Blend overlay
    result = cv2.addWeighted(overlay, 0.35, img, 0.65, 0)
    return result


# Show 6 random training samples
train_images_list = sorted(glob.glob(os.path.join(TRAIN_IMAGES, "*.jpg")))
samples = random.sample(train_images_list, min(6, len(train_images_list)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.flat, samples):
    basename = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(TRAIN_LABELS, basename + ".txt")
    vis = draw_yolo_polygons(img_path, lbl_path)
    ax.imshow(vis)
    ax.set_title(basename[:30], fontsize=9)
    ax.axis("off")
plt.suptitle("Training Samples with Ground Truth Polygons", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 5 - Create data.yaml for YOLO Training
# ============================================================

# Read the original data.yaml to get class names
original_yaml = os.path.join(DATA_ROOT, "data.yaml")
if os.path.exists(original_yaml):
    with open(original_yaml, "r") as f:
        orig_cfg = yaml.safe_load(f)
    print("Original data.yaml:")
    print(yaml.dump(orig_cfg, default_flow_style=False))
    class_names = orig_cfg.get("names", {0: "pothole"})
    nc = orig_cfg.get("nc", 1)
else:
    class_names = {0: "pothole"}
    nc = 1
    print("No data.yaml found, using defaults.")

# Write a new data.yaml with absolute paths for training
data_yaml_path = "/kaggle/working/data.yaml"
data_cfg = {
    "path": DATA_ROOT,
    "train": "train/images",
    "val": "val/images",
    "nc": nc,
    "names": class_names,
}

with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"\nTraining data.yaml written to: {data_yaml_path}")
with open(data_yaml_path, "r") as f:
    print(f.read())

In [ ]:
# ============================================================
# Cell 6 - Load Pretrained YOLOv8-seg Model
# ============================================================

# YOLOv8n-seg: nano segmentation model (fast, good for learning)
model = YOLO("yolov8n-seg.pt")

print(f"Model: {model.model_name}")
print(f"Task:  {model.task}")

# Quick architecture summary
print(f"\nModel loaded successfully. Ready for segmentation training.")

In [ ]:
# ============================================================
# Cell 7 - Train the Model
# ============================================================

# Training hyperparameters
EPOCHS   = 30
IMG_SIZE = 640
BATCH    = 8

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=10,         # Early stopping patience
    save=True,
    save_period=10,      # Save checkpoint every 10 epochs
    device=0,            # GPU
    workers=2,
    seed=SEED,
    pretrained=True,
    optimizer="Adam",
    lr0=1e-3,
    lrf=0.01,            # Final LR = lr0 * lrf
    augment=True,
    project="/kaggle/working/runs",
    name="pothole_seg",
    exist_ok=True,
)

print("\nTraining complete!")

In [ ]:
# ============================================================
# Cell 8 - Training Curves
# ============================================================

# Ultralytics saves training plots automatically
results_dir = "/kaggle/working/runs/pothole_seg"

# Display the auto-generated results plot
results_png = os.path.join(results_dir, "results.png")
if os.path.exists(results_png):
    display(Image(filename=results_png, width=900))
else:
    print("results.png not found, showing CSV data instead.")

# Also show confusion matrix if available
cm_path = os.path.join(results_dir, "confusion_matrix.png")
if os.path.exists(cm_path):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_path, width=600))

In [ ]:
# ============================================================
# Cell 9 - Evaluate on Validation Set
# ============================================================

# Load the best model from training
best_model_path = os.path.join(results_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)

# Run validation
metrics = best_model.val(
    data=data_yaml_path,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    split="val",
)

# Print key metrics
print("\n" + "=" * 50)
print("Validation Results (Segmentation)")
print("=" * 50)
print(f"  mAP50  (mask): {metrics.seg.map50:.4f}")
print(f"  mAP50-95 (mask): {metrics.seg.map:.4f}")
print(f"  mAP50  (box):  {metrics.box.map50:.4f}")
print(f"  mAP50-95 (box):  {metrics.box.map:.4f}")

In [ ]:
# ============================================================
# Cell 10 - Visualize Predictions on Validation Images
# ============================================================

# Run inference on random validation images
val_images_list = sorted(glob.glob(os.path.join(VAL_IMAGES, "*.jpg")))
sample_val = random.sample(val_images_list, min(8, len(val_images_list)))

preds = best_model.predict(
    source=sample_val,
    imgsz=IMG_SIZE,
    conf=0.25,
    device=0,
    save=False,
)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, result in zip(axes.flat, preds):
    # result.plot() returns BGR numpy array with masks drawn
    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    # Show detection count
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f"{n_det} pothole(s) detected", fontsize=10)
    ax.axis("off")

plt.suptitle("YOLOv8-seg Predictions on Validation Images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 11 - Detailed Mask Visualization
# ============================================================

def visualize_masks_detail(model, image_paths, n=4):
    """Show original image, predicted mask, and overlay side by side."""
    samples = random.sample(image_paths, min(n, len(image_paths)))
    results = model.predict(source=samples, imgsz=IMG_SIZE, conf=0.25,
                            device=0, save=False, verbose=False)

    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for row, (img_path, result) in enumerate(zip(samples, results)):
        # Original image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Build combined mask
        h, w = img.shape[:2]
        mask_combined = np.zeros((h, w), dtype=np.uint8)
        if result.masks is not None:
            for m in result.masks.data:
                mask_np = m.cpu().numpy()
                mask_resized = cv2.resize(mask_np, (w, h))
                mask_combined = np.maximum(mask_combined,
                                           (mask_resized > 0.5).astype(np.uint8) * 255)

        # Overlay
        overlay = img.copy()
        overlay[mask_combined > 0] = (
            overlay[mask_combined > 0] * 0.5 +
            np.array([255, 0, 0]) * 0.5
        ).astype(np.uint8)

        axes[row, 0].imshow(img)
        axes[row, 0].set_title("Original")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask_combined, cmap="gray")
        axes[row, 1].set_title("Predicted Mask")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(overlay)
        axes[row, 2].set_title("Overlay")
        axes[row, 2].axis("off")

    plt.suptitle("Detailed Segmentation Results", fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_masks_detail(best_model, val_images_list, n=4)

In [ ]:
# ============================================================
# Cell 12 - Save Model
# ============================================================

import shutil

# Copy best weights to a clean output path
output_path = "/kaggle/working/yolov8n_pothole_seg_best.pt"
shutil.copy2(best_model_path, output_path)

print(f"Best model saved to: {output_path}")
print(f"Model size: {os.path.getsize(output_path) / 1e6:.1f} MB")

# Summary
print("\n" + "=" * 50)
print("Training Summary")
print("=" * 50)
print(f"  Architecture:   YOLOv8n-seg")
print(f"  Dataset:        Pothole Image Segmentation")
print(f"  Train images:   {n_train_img}")
print(f"  Val images:     {n_val_img}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Image size:     {IMG_SIZE}")
print(f"  Mask mAP50:     {metrics.seg.map50:.4f}")
print(f"  Mask mAP50-95:  {metrics.seg.map:.4f}")
print(f"  Output:         {output_path}")